In [23]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:16pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [24]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import time

%config InlineBackend.figure_format = "retina"
sns.set(style='white', context='notebook', palette='Dark2',
        rc={'figure.figsize': (12,3), 'font.family':'Malgun Gothic', 'axes.unicode_minus':False})

import warnings
warnings.filterwarnings(action='ignore')
pd.set_option('display.max_columns', None)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, roc_curve


In [25]:
df = pd.read_csv('00_data/dataset_features.csv', low_memory=False)
print(df.shape)      

(254750, 164)


In [26]:
X_label = pd.read_csv('00_data/feature_cols.csv')['파생변수'].tolist()
y_label = '폐업위기_y1'

# 3절. 데이터 분할 (시간순)
- train_data : 2023Q1–2024Q4 / valid_data : 2025Q1–Q2 / test_data : 2025Q3 

In [27]:
train_data = df[df['기준_년분기_코드'] <= 20244]
valid_data = df[(df['기준_년분기_코드'] >= 20251) & (df['기준_년분기_코드'] <= 20252)]

X_train, X_val, y_train, y_val = train_data[X_label], valid_data[X_label], train_data[y_label], valid_data[y_label]

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((170738, 28), (41759, 28), (170738,), (41759,))

In [28]:
print('train :', X_train.shape, ' 위기 비율', round(y_train.mean() * 100, 1), '%')
print('valid :', X_val.shape, ' 위기 비율', round(y_val.mean() * 100, 1), '%')

train : (170738, 28)  위기 비율 20.3 %
valid : (41759, 28)  위기 비율 19.3 %


## 3.1 XGBoost용 불균형 가중치  (삭제 검토필요)
- scale_pos_weight = 정상 행 수 ÷ 위기 행 수

In [29]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight :', round(spw, 2))

scale_pos_weight : 3.92


# 4절. 모델 정의 (H0 기본값)

In [34]:
## 4.1 B1 로지스틱 회귀 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
X_train_oh = pd.get_dummies(X_train, columns=['상권구분', '상권변화지표', '업종'], dtype=int)
X_val_oh = pd.get_dummies(X_val, columns=['상권구분', '상권변화지표', '업종'], dtype=int)
X_val_oh = X_val_oh.reindex(columns=X_train_oh.columns, fill_value=0)   # train과 컬럼 맞춤

print('원핫인코딩 전 :', X_train.shape[1], '원핫인코딩 후 :', X_train_oh.shape[1])

원핫인코딩 전 : 28 원핫인코딩 후 : 96


In [35]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_oh)  
X_valid_scaled = scaler.transform(X_val_oh)      

In [37]:
lr_model = LogisticRegression(C=1.0,                  
                              class_weight='balanced',  
                              max_iter=1000)

In [38]:
lr_model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [46]:
# B1검증
위기확률_lr = lr_model.predict_proba(X_valid_scaled)[:, 1]
예측값_lr = (위기확률_lr >= 0.5).astype(int)    

auc = roc_auc_score(y_val, 위기확률_lr)
rec = recall_score(y_val, 예측값_lr)
pre = precision_score(y_val, 예측값_lr)
f1  = f1_score(y_val, 예측값_lr)
print(f'AUC {auc:.4f} | 재현율 {rec:.4f} | 정밀도 {pre:.4f} | F1 {f1:.4f}')

AUC 0.7673 | 재현율 0.7010 | 정밀도 0.3634 | F1 0.4787


In [51]:
record = []
record.append(['E01', 1, 'A0(원핫)', X_train_oh.shape[1], 'B1 로지스틱', 'H0', y_label,
               round(auc, 4), round(rec, 4), round(pre, 4), round(f1, 4)])
# cols = ['실험ID', '단계', '변수군', '변수수', '모델', 'H', 'y',
#         'valid_AUC', '재현율', '정밀도', 'F1', '학습시간(초)']

In [52]:
record

[['E01',
  1,
  'A0(원핫)',
  96,
  'B1 로지스틱',
  'H0',
  '폐업위기_y1',
  0.7673,
  0.701,
  0.3634,
  0.4787]]

In [54]:
## 4.2 B2 의사결정나무 정의
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=6,              # 최대 나무 깊이
                                  min_samples_leaf=50,      # 잎 노드의 최소 데이터 수 (과적합 방지)
                                  class_weight='balanced',  # 위기(1)가 적어서 가중치 자동 조정
                                  )

In [55]:
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=6,
                       min_samples_leaf=50)

In [56]:
위기확률_dt = dt_model.predict_proba(X_val)[:, 1]        # 위기(1)일 확률
예측값_dt = (위기확률_dt >= 0.5).astype(int)              # 0.5 기준 0/1 예측

auc = roc_auc_score(y_val, 위기확률_dt)
rec = recall_score(y_val, 예측값_dt)
pre = precision_score(y_val, 예측값_dt)
f1  = f1_score(y_val, 예측값_dt)
print(f'AUC {auc:.4f} | 재현율 {rec:.4f} | 정밀도 {pre:.4f} | F1 {f1:.4f}')

record.append({'실험ID': 'E02',
               '단계': 1,
               '변수군': 'A0',
               '변수수': X_train.shape[1],
               '모델': 'B2 의사결정나무',
               'H': 'H0',
               'y': y_label,
               'valid_AUC': round(auc, 4),
               '재현율': round(rec, 4),
               '정밀도': round(pre, 4),
               'F1': round(f1, 4)})

AUC 0.7365 | 재현율 0.7265 | 정밀도 0.3178 | F1 0.4422


In [58]:
!pip list --format=freeze > requirements.txt